## Clean metadata

In [1]:
import gzip
import json
import gc
from pathlib import Path

import pandas as pd
import numpy as np
import duckdb

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
META_TEMP_DIR = Path("../data/temp_metadata")

META_TEMP_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / "meta_Electronics.json.gz"
FILTERED_PATH = PROCESSED_DIR / "reviews_filtered_min5.parquet"

In [3]:
def read_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

In [5]:
def clean_meta_row(row):
    asin = row.get("asin")

    if asin is None:
        return None

    title = row.get("title", "")
    brand = row.get("brand", "")
    description = row.get("description", "")
    feature = row.get("feature", [])
    category = row.get("category", [])
    price = row.get("price")
    also_buy = row.get("also_buy", [])
    also_view = row.get("also_view", [])

    if not isinstance(title, str):
        title = ""

    if not isinstance(brand, str):
        brand = ""

    if isinstance(description, list):
        description = " ".join([str(x) for x in description])
    elif not isinstance(description, str):
        description = ""

    if isinstance(feature, list):
        feature_text = " ".join([str(x) for x in feature])
    else:
        feature_text = ""

    if isinstance(category, list):
        category_text = " > ".join([str(x) for x in category])
        main_category = category[0] if len(category) > 0 else ""
        sub_category = category[-1] if len(category) > 0 else ""
    else:
        category_text = ""
        main_category = ""
        sub_category = ""

    parsed_price = None

    if price is not None:
        try:
            price_str = str(price)
            price_str = price_str.replace("$", "").replace(",", "").strip()
            parsed_price = float(price_str)

            if parsed_price < 0:
                parsed_price = None
        except:
            parsed_price = None

    if not isinstance(also_buy, list):
        also_buy = []

    if not isinstance(also_view, list):
        also_view = []

    title = title.strip().lower()
    brand = brand.strip().lower()
    description = description.strip().lower()
    feature_text = feature_text.strip().lower()
    category_text = category_text.strip().lower()
    main_category = str(main_category).strip().lower()
    sub_category = str(sub_category).strip().lower()

    text_all = " ".join([
        title,
        brand,
        category_text,
        feature_text,
        description
    ]).strip()

    return {
        "item_id": asin,
        "title": title,
        "brand": brand,
        "description": description,
        "feature_text": feature_text,
        "category_text": category_text,
        "main_category": main_category,
        "sub_category": sub_category,
        "price": parsed_price,
        "also_buy_count": len(also_buy),
        "also_view_count": len(also_view),
        "content_text": text_all,
    }

In [6]:
def clean_metadata_to_parquet(path, out_dir, chunk_size=100_000):
    buffer = []
    chunk_id = 0
    total_read = 0
    total_saved = 0

    for row in read_json_gz(path):
        total_read += 1

        cleaned = clean_meta_row(row)

        if cleaned is not None:
            buffer.append(cleaned)

        if len(buffer) >= chunk_size:
            df = pd.DataFrame(buffer)

            out_path = out_dir / f"metadata_clean_chunk_{chunk_id:04d}.parquet"
            df.to_parquet(out_path, index=False)

            total_saved += len(df)

            print(
                f"Saved metadata chunk {chunk_id:04d} | "
                f"rows: {len(df):,} | "
                f"total read: {total_read:,} | "
                f"total saved: {total_saved:,}"
            )

            buffer.clear()
            del df
            gc.collect()

            chunk_id += 1

    if buffer:
        df = pd.DataFrame(buffer)

        out_path = out_dir / f"metadata_clean_chunk_{chunk_id:04d}.parquet"
        df.to_parquet(out_path, index=False)

        total_saved += len(df)

        print(
            f"Saved final metadata chunk {chunk_id:04d} | "
            f"rows: {len(df):,} | "
            f"total read: {total_read:,} | "
            f"total saved: {total_saved:,}"
        )

        buffer.clear()
        del df
        gc.collect()

    print("DONE")
    print("Total read:", total_read)
    print("Total saved:", total_saved)

In [7]:
clean_metadata_to_parquet(
    META_PATH,
    META_TEMP_DIR,
    chunk_size=100_000
)

Saved metadata chunk 0000 | rows: 100,000 | total read: 100,000 | total saved: 100,000
Saved metadata chunk 0001 | rows: 100,000 | total read: 200,000 | total saved: 200,000
Saved metadata chunk 0002 | rows: 100,000 | total read: 300,000 | total saved: 300,000
Saved metadata chunk 0003 | rows: 100,000 | total read: 400,000 | total saved: 400,000
Saved metadata chunk 0004 | rows: 100,000 | total read: 500,000 | total saved: 500,000
Saved metadata chunk 0005 | rows: 100,000 | total read: 600,000 | total saved: 600,000
Saved metadata chunk 0006 | rows: 100,000 | total read: 700,000 | total saved: 700,000
Saved final metadata chunk 0007 | rows: 86,445 | total read: 786,445 | total saved: 786,445
DONE
Total read: 786445
Total saved: 786445


Giữ metadata của item còn lại

In [4]:
con = duckdb.connect()

META_CHUNKS = str(META_TEMP_DIR / "metadata_clean_chunk_*.parquet")
ITEMS_PATH = PROCESSED_DIR / "items_clean_min5.parquet"

In [9]:
con.execute(f"""
COPY (
    SELECT DISTINCT m.*
    FROM read_parquet('{META_CHUNKS}') m
    INNER JOIN (
        SELECT DISTINCT item_id
        FROM read_parquet('{FILTERED_PATH}')
    ) r
    ON m.item_id = r.item_id
) TO '{ITEMS_PATH}' (FORMAT PARQUET)
""")

In [10]:
## kiểm tra
items_stats = con.execute(f"""
SELECT
    COUNT(*) AS metadata_items,
    COUNT(DISTINCT item_id) AS unique_items,
    SUM(CASE WHEN title IS NULL OR title = '' THEN 1 ELSE 0 END) AS missing_title,
    SUM(CASE WHEN brand IS NULL OR brand = '' THEN 1 ELSE 0 END) AS missing_brand,
    SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END) AS missing_description,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price
FROM read_parquet('{ITEMS_PATH}')
""").df()

items_stats

,metadata_items,unique_items,missing_title,missing_brand,missing_description,missing_price
0,283940,283940,6.0,1229.0,39414.0,159866.0


Metadata coverage

In [11]:
coverage = con.execute(f"""
SELECT
    r.review_items,
    i.metadata_items,
    i.metadata_items * 100.0 / r.review_items AS metadata_coverage_percent
FROM
    (
        SELECT COUNT(DISTINCT item_id) AS review_items
        FROM read_parquet('{FILTERED_PATH}')
    ) r,
    (
        SELECT COUNT(DISTINCT item_id) AS metadata_items
        FROM read_parquet('{ITEMS_PATH}')
    ) i
""").df()

coverage

,review_items,metadata_items,metadata_coverage_percent
0,284114,283940,99.938757


Merge interaction + metadata

In [12]:
MERGED_PATH = PROCESSED_DIR / "reviews_merged_min5.parquet"
con.execute(f"""
COPY (
    SELECT
        r.user_id,
        r.item_id,
        r.rating,
        r.review_text,
        r.summary,
        r.verified,
        r.vote,
        r.helpful_score,
        r.timestamp,

        m.title,
        m.brand,
        m.description,
        m.feature_text,
        m.category_text,
        m.main_category,
        m.sub_category,
        m.price,
        m.also_buy_count,
        m.also_view_count,
        m.content_text
    FROM read_parquet('{FILTERED_PATH}') r
    LEFT JOIN read_parquet('{ITEMS_PATH}') m
    ON r.item_id = m.item_id
) TO '{MERGED_PATH}' (FORMAT PARQUET)
""")

In [13]:
# kiểm tra
merged_stats = con.execute(f"""
SELECT
    COUNT(*) AS interactions,
    COUNT(DISTINCT user_id) AS users,
    COUNT(DISTINCT item_id) AS items,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS rows_without_metadata
FROM read_parquet('{MERGED_PATH}')
""").df()

merged_stats

,interactions,users,items,rows_without_metadata
0,7059313,788112,284114,3696.0


## Temporal split

Vì dữ liệu từ 1999-06-13 đến 2018-10-04 nên sẽ dc chia như sau:

Train: trước 2017-01-01

Validation: 2017-01-01 đến trước 2018-01-01

Test: từ 2018-01-01 trở đi

In [6]:
TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VAL_PATH = PROCESSED_DIR / "val.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

In [15]:
con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{MERGED_PATH}')
    WHERE timestamp < DATE '2017-01-01'
) TO '{TRAIN_PATH}' (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{MERGED_PATH}')
    WHERE timestamp >= DATE '2017-01-01'
      AND timestamp < DATE '2018-01-01'
) TO '{VAL_PATH}' (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{MERGED_PATH}')
    WHERE timestamp >= DATE '2018-01-01'
) TO '{TEST_PATH}' (FORMAT PARQUET)
""")

In [ ]:
# kiểm tra split
split_stats = con.execute(f"""
SELECT 'train' AS split,
       COUNT(*) AS interactions,
       COUNT(DISTINCT user_id) AS users,
       COUNT(DISTINCT item_id) AS items,
       MIN(timestamp) AS min_time,
       MAX(timestamp) AS max_time
FROM read_parquet('{TRAIN_PATH}')

UNION ALL

SELECT 'val' AS split,
       COUNT(*) AS interactions,
       COUNT(DISTINCT user_id) AS users,
       COUNT(DISTINCT item_id) AS items,
       MIN(timestamp) AS min_time,
       MAX(timestamp) AS max_time
FROM read_parquet('{VAL_PATH}')

UNION ALL

SELECT 'test' AS split,
       COUNT(*) AS interactions,
       COUNT(DISTINCT user_id) AS users,
       COUNT(DISTINCT item_id) AS items,
       MIN(timestamp) AS min_time,
       MAX(timestamp) AS max_time
FROM read_parquet('{TEST_PATH}')
""").df()

split_stats

# 270719

,split,interactions,users,items,min_time,max_time
0,train,5677563,752848,270719,1999-06-13,2016-12-31
1,val,975996,342154,121626,2017-01-01,2017-12-31
2,test,405754,182894,70749,2018-01-01,2018-10-04


In [17]:
# check cold-start
cold_start_stats = con.execute(f"""
WITH train_users AS (
    SELECT DISTINCT user_id
    FROM read_parquet('{TRAIN_PATH}')
),
train_items AS (
    SELECT DISTINCT item_id
    FROM read_parquet('{TRAIN_PATH}')
),
val_data AS (
    SELECT *
    FROM read_parquet('{VAL_PATH}')
),
test_data AS (
    SELECT *
    FROM read_parquet('{TEST_PATH}')
)

SELECT
    'val' AS split,
    COUNT(*) AS interactions,
    SUM(CASE WHEN tu.user_id IS NULL THEN 1 ELSE 0 END) AS cold_user_rows,
    SUM(CASE WHEN ti.item_id IS NULL THEN 1 ELSE 0 END) AS cold_item_rows
FROM val_data v
LEFT JOIN train_users tu ON v.user_id = tu.user_id
LEFT JOIN train_items ti ON v.item_id = ti.item_id

UNION ALL

SELECT
    'test' AS split,
    COUNT(*) AS interactions,
    SUM(CASE WHEN tu.user_id IS NULL THEN 1 ELSE 0 END) AS cold_user_rows,
    SUM(CASE WHEN ti.item_id IS NULL THEN 1 ELSE 0 END) AS cold_item_rows
FROM test_data t
LEFT JOIN train_users tu ON t.user_id = tu.user_id
LEFT JOIN train_items ti ON t.item_id = ti.item_id
""").df()

cold_start_stats

,split,interactions,cold_user_rows,cold_item_rows
0,val,975996,163202.0,33225.0
1,test,405754,79767.0,24583.0


## lưu kết quả phase 2.b

In [18]:
phase2_final_summary = {
    "filtered_interactions": 7059313,
    "filtered_users": 788112,
    "filtered_items": 284114,
    "filtered_sparsity": 0.999968,
    "min_user_interactions": 5,
    "min_item_interactions": 5,
}

phase2_final_df = pd.DataFrame([phase2_final_summary])
phase2_final_df.to_csv(PROCESSED_DIR / "phase2_final_summary.csv", index=False)

phase2_final_df

,filtered_interactions,filtered_users,filtered_items,filtered_sparsity,min_user_interactions,min_item_interactions
0,7059313,788112,284114,0.999968,5,5


## TẠO WARM-VALIDATION

In [9]:
WARM_VAL_PATH = PROCESSED_DIR / "val_warm.parquet"

con.execute(f"""
COPY (
    WITH train_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{TRAIN_PATH}')
    ),
    train_items AS (
        SELECT DISTINCT item_id
        FROM read_parquet('{TRAIN_PATH}')
    )

    SELECT v.*
    FROM read_parquet('{VAL_PATH}') v
    INNER JOIN train_users tu
        ON v.user_id = tu.user_id
    INNER JOIN train_items ti
        ON v.item_id = ti.item_id
) TO '{WARM_VAL_PATH}' (FORMAT PARQUET)
""")

## Warm test

In [10]:
WARM_TEST_PATH = PROCESSED_DIR / "test_warm.parquet"

con.execute(f"""
COPY (
    WITH train_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{TRAIN_PATH}')
    ),
    train_items AS (
        SELECT DISTINCT item_id
        FROM read_parquet('{TRAIN_PATH}')
    )

    SELECT t.*
    FROM read_parquet('{TEST_PATH}') t
    INNER JOIN train_users tu
        ON t.user_id = tu.user_id
    INNER JOIN train_items ti
        ON t.item_id = ti.item_id
) TO '{WARM_TEST_PATH}' (FORMAT PARQUET)
""")

In [2]:
import duckdb

df = duckdb.query("""
    SELECT *
    FROM 'E:/DACNTT_nopbai/source_code/data/processed/train.parquet'
    LIMIT 5
""").to_df()

print(df)

          user_id     item_id  rating  \
0  A1A8JRGU38PU02  B0015M22C6     5.0   
1  A2PREU4LOFQRB1  B0015M22C6     5.0   
2  A25WIE7H1K61ZK  B0015M22C6     5.0   
3  A19X3Y9J2EYCHF  B0015M22C6     5.0   
4  A3RCY0OBPZ5XWF  B0015M22C6     5.0   

                                         review_text  \
0  i should have purchased this sooner. great pro...   
1  first let me explain why i needed this product...   
2  this is the most compact enclosure i own. it i...   
3  the vantec nexstar cx is a great, inexpensive ...   
4  my screen died on my old laptop, i knew the co...   

                                           summary  verified  vote  \
0                                    great product         1     0   
1          very nice low cost hard drive enclosure         1    88   
2  compact and stays cool; capacity actually 500gb         1     0   
3   3 of 'em in use for 5 to 18 months w/o a hitch         1     0   
4                                       life saver         1     0

In [6]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df = duckdb.query("""
    SELECT *
    FROM 'E:/DACNTT_nopbai/source_code/data/processed/model_ready/train_model.parquet'
    LIMIT 5
""").to_df()

print(df)

   user_idx  item_idx  is_known_user  is_known_item  is_warm_row  \
0    597348      4355              1              1            1   
1    542027      4355              1              1            1   
2    453250      4355              1              1            1   
3    137557      4247              1              1            1   
4    274024      4247              1              1            1   

          user_id     item_id  rating  item_quality  item_value  item_design  \
0   A7RFPVB461HZ7  B000067SBL     5.0      4.174949    4.000714     3.334440   
1  A3Q0W8EKTMEGKR  B000067SBL     4.0      4.174949    4.000714     3.334440   
2  A39YKHUAZYXTG8  B000067SBL     5.0      4.174949    4.000714     3.334440   
3  A1OR1K4XSK5OV0  B000067O6B     2.0      3.469914    2.767408     2.928061   
4  A2DG1NQM08X1EI  B000067O6B     3.0      3.469914    2.767408     2.928061   

   item_usability  item_durability  mask_quality  mask_value  mask_design  \
0        3.748365         3.68715